# Tiny Shakespeare dataset walkthrough

This notebook loads the local `tiny_shakespeare.py` dataset builder, shows a sample of the raw text, and then turns the text into a character-tokenized dataset suitable for next-character prediction.

In [1]:
from datasets import Dataset, load_dataset

/Users/matt/Library/CloudStorage/OneDrive-Personal/Code/LLM/BitNet6502/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("modelling/tiny_shakespeare.py")
dataset

RuntimeError: Dataset scripts are no longer supported, but found tiny_shakespeare.py

In [ ]:
for split_name, split in dataset.items():
    text = split[0]["text"]
    print(f"{split_name}: {len(text):,} characters")
    print(text[:500])
    print("\n" + "-" * 80 + "\n")

## Character tokenization

The dataset builder returns one long string per split. For character-level language modelling, we can:

1. Build a vocabulary from the training split.
2. Map each character to an integer id.
3. Slice the integer stream into fixed-length training examples.
4. Predict the next character at every position.

In [ ]:
train_text = dataset["train"][0]["text"]

vocab = sorted(set(train_text))
stoi = {ch: idx for idx, ch in enumerate(vocab)}
itos = {idx: ch for ch, idx in stoi.items()}

def encode(text: str) -> list[int]:
    return [stoi[ch] for ch in text]


def decode(token_ids: list[int]) -> str:
    return "".join(itos[idx] for idx in token_ids)


encoded_train = encode(train_text)

print(f"Vocabulary size: {len(vocab)}")
print(f"First 20 vocabulary items: {vocab[:20]}")
print(f"First 80 chars: {train_text[:80]!r}")
print(f"First 80 token ids: {encoded_train[:80]}")

In [ ]:
block_size = 64
stride = 64

examples = []
for start in range(0, len(encoded_train) - block_size, stride):
    x = encoded_train[start : start + block_size]
    y = encoded_train[start + 1 : start + block_size + 1]
    examples.append(
        {
            "input_ids": x,
            "labels": y,
            "input_text": decode(x),
            "target_text": decode(y),
        }
    )

char_dataset = Dataset.from_list(examples)
char_dataset

In [ ]:
sample = char_dataset[0]
print("input_ids:", sample["input_ids"])
print("labels   :", sample["labels"])
print()
print("input_text:")
print(repr(sample["input_text"]))
print()
print("target_text:")
print(repr(sample["target_text"]))

In [ ]:
preview_rows = [char_dataset[i] for i in range(3)]
for idx, row in enumerate(preview_rows):
    print(f"Example {idx}")
    print("input_text :", repr(row["input_text"]))
    print("target_text:", repr(row["target_text"]))
    print()

This `char_dataset` is now in a shape you can feed into a character-level model: `input_ids` are the context window and `labels` are the same sequence shifted by one character.